In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime

In [8]:
def scrape_autoscout(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all car listings on the page
    cars = soup.find_all('article', class_='cldt-summary-full-item')

    # Prepare a list to store the car data
    car_data = []

    for car in cars:
        # Extract the data from each car listing
        url = car.find('a', class_="ListItem_title__ndA4s")['href']
        url = 'https://www.autoscout24.es' + url
        location = car.find('span', class_="SellerInfo_address__leRMu")
        if location is None:
            location = car.find('span', class_="SellerInfo_private__THzvQ").text
        else:
            location = location.text

        title_element = car.find('h2')
        description = title_element.find('span', class_='ListItem_version__5EWfi').text.strip()
        title = title_element.text.replace(description, '').strip()
        brand = title.split()[0]

        rating_elem = car.find('div', class_="scr-price-label")
        rating = rating_elem.find('p').text
        price = car.find('p', class_='Price_price__APlgs').text.strip()
        price = int(re.sub(r'\D', '', price))

        details_elements = car.find_all('span', class_='VehicleDetailTable_item__4n35N')
        details = [detail.text.strip() for detail in details_elements]
        mileage = details[0]
        mileage = int(re.sub(r'\D', '', mileage))
        transmission = details[1]
        year = details[2]
        # year = datetime.strptime(year, '%m/%Y')
        engine_type = details[3]
        cv = details[4]
        cv = int(re.search(r'(\d+) CV', cv).group(1))

        # Add the data to our list
        car_data.append([brand, title, description, rating, price, mileage, transmission, year, engine_type, cv, location, url])

    # Convert the list to a DataFrame and return it
    return pd.DataFrame(car_data, columns=['brand', 'title', 'description', 'rating', 'full_price', 'mileage', 'transmission', 'year', 'fuel', 'cv', 'location', 'url']
)

def scrape_multiple_pages(base_url):
    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find max page 
    page = soup.find('li', class_='pagination-item--disabled pagination-item--page-indicator')
               
    if page is None:
        return
    else:
        page= int(page.text.split()[-1])

    # Prepare a list to store the DataFrames
    dfs = []

    for i in range(1, page + 1):
        # Construct the URL for the current page
        url = base_url + '&page=' + str(i)

        # Scrape the current page and store the DataFrame
        df = scrape_autoscout(url)
        dfs.append(df)

    # Concatenate all the DataFrames
    master_df = pd.concat(dfs, ignore_index=True)

    return master_df


In [9]:
df = pd.DataFrame()
for i in range(70):
    url = f'https://www.autoscout24.es/lst?atype=C&body=4%2C5%2C12&cy=E&damaged_listing=exclude&desc=0&fregfrom=2017&kmfrom=20000&kmto=100000&lat=40.41669&lon=-3.70035&powerfrom=59&powertype=hp&pricefrom={(i+30)*2}00&priceto={(i+31)*2}00&search_id=49cs04bi9u&sort=price&source=homepage_search-mask&ustate=N%2CU&zip=madrid-%28comunidad-de-madrid%29&zipr=50'
    df = pd.concat([df, scrape_multiple_pages(url)],ignore_index=True)
    df = df.drop_duplicates()
    print(df.shape)
    
df.to_csv(f'data/autoscout/{datetime.now().strftime("%y.%m.%d")}_autoscout_raw.csv', index=False)

(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(0, 0)
(2, 12)
(2, 12)
(3, 12)
(8, 12)
(8, 12)
(8, 12)
(9, 12)
(11, 12)
(19, 12)
(20, 12)
(21, 12)
(33, 12)
(41, 12)
(67, 12)
(74, 12)
(89, 12)
(124, 12)
(155, 12)
(203, 12)
(225, 12)
(261, 12)
(313, 12)
(359, 12)
(438, 12)
(495, 12)
(553, 12)
(675, 12)
(719, 12)
(866, 12)
(925, 12)
(1017, 12)
(1152, 12)
(1243, 12)
(1457, 12)
(1536, 12)
(1616, 12)
(1775, 12)
(1844, 12)
(2089, 12)
(2154, 12)
(2344, 12)
(2465, 12)
(2605, 12)
(2950, 12)
(3024, 12)
(3126, 12)
(3278, 12)
(3426, 12)
(3659, 12)
(3757, 12)
(3834, 12)
(4025, 12)
(4105, 12)
(4408, 12)
(4469, 12)
(4554, 12)
(4725, 12)
(4811, 12)
(5065, 12)
